# ATP Data Exploratory Data Analysis (EDA)

This notebook is prepared to explore ATP match data in the `datas/` folder and extract summaries for presentation.

Objectives:
- How many total rows (records) and variables (columns) are there?
- What is the distribution of data types?
- What is the general distribution of missing values?
- Basic categorical distributions (surface, level, round, etc.)
- Basic numerical distributions (minutes, age, height, etc.)
- Time coverage (number of matches by year)


In [ ]:
# Imports and settings
from pathlib import Path
import json
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
sns.set(style="whitegrid", context="notebook")
pd.set_option('display.max_columns', 100)
DATA_DIR = Path('../datas').resolve()
DATA_DIR


In [ ]:
# File discovery: find atp_matches_*.csv files
csv_files = sorted([p for p in DATA_DIR.glob('atp_matches_*.csv')])
len(csv_files), csv_files[:5]


In [ ]:
# Summary of file count by year
def year_from_name(p: Path):
    try:
        return int(p.stem.split('_')[-1])
    except Exception:
        return None
years = [year_from_name(p) for p in csv_files]
summary_years = pd.Series(years).value_counts().sort_index()
summary_years.head(), summary_years.tail(), (summary_years.index.min(), summary_years.index.max())


In [ ]:
# Extract column set with meta reading (fast)
all_cols = set()
sample_cols = {}
for p in csv_files[:10]:  # sample columns from first few files
    try:
        df0 = pd.read_csv(p, nrows=0, low_memory=False)
        all_cols.update(df0.columns.tolist())
        sample_cols[p.name] = df0.columns.tolist()
    except Exception as e:
        print('Error:', p.name, e)
len(all_cols), list(sorted(all_cols))[:20]


In [ ]:
# Quickly calculate total row count with chunks (RAM friendly)
def count_rows(files, chunksize=200_000):
    total = 0
    for p in files:
        try:
            for chunk in pd.read_csv(p, chunksize=chunksize, low_memory=False):
                total += len(chunk)
        except Exception as e:
            print('Error counting:', p.name, e)
    return total
total_rows = count_rows(csv_files)
total_cols = len(all_cols)
print('Total Records (rows):', total_rows)
print('Total Variables (columns):', total_cols)


In [ ]:
# Data types: dtype inference with sample concatenation (sampling)
def sample_concat(files, max_files=5, max_rows_per_file=50_000):
    dfs = []
    for p in files[:max_files]:
        try:
            df = pd.read_csv(p, nrows=max_rows_per_file, low_memory=False)
            dfs.append(df)
        except Exception as e:
            print('Error reading:', p.name, e)
    if dfs:
        return pd.concat(dfs, ignore_index=True)
    return pd.DataFrame()
sample_df = sample_concat(csv_files, max_files=6, max_rows_per_file=60_000)
sample_df.shape, sample_df.head(2)


In [ ]:
# Dtype summary
dtype_summary = sample_df.dtypes.value_counts().rename_axis('dtype').reset_index(name='count')
dtype_summary


In [ ]:
# Missing value analysis (in sample)
na_counts = sample_df.isna().sum().sort_values(ascending=False)
na_pct = (sample_df.isna().mean() * 100).sort_values(ascending=False)
na_df = pd.DataFrame({'na_count': na_counts, 'na_pct': na_pct})
na_df.head(20)


In [ ]:
# Categorical summaries (if available): surface, tourney_level, round
for col in ['surface', 'tourney_level', 'round']:
    if col in sample_df.columns:
        display(pd.DataFrame(sample_df[col].value_counts().head(10)).rename(columns={col: 'count'}))
    else:
        print(f'Column not found: {col}')


In [ ]:
# Numerical distributions (if available): minutes, winner_age, loser_age, winner_ht, loser_ht
num_cols = [c for c in ['minutes', 'winner_age', 'loser_age', 'winner_ht', 'loser_ht'] if c in sample_df.columns]
fig, axes = plt.subplots(len(num_cols), 1, figsize=(8, 3*max(1, len(num_cols))), constrained_layout=True)
if len(num_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, num_cols):
    sns.histplot(sample_df[col].dropna(), bins=50, ax=ax, kde=False)
    ax.set_title(f'Distribution: {col}')
fig.suptitle('Numerical Variable Distributions', y=1.02)
plt.show()


In [ ]:
# Time coverage: number of matches by year (based on file name)
year_counts = pd.Series([y for y in years if y is not None]).value_counts().sort_index()
ax = year_counts.plot(kind='bar', figsize=(14,4), color='#1f77b4')
ax.set_title('File (and approximate match) count by Year')
ax.set_xlabel('Year')
ax.set_ylabel('File count')
plt.show()


## Visualizations and simple insights from sample
This section produces quick visualizations and a few basic insights on `sample_df`. Plots are for the sample (not all data).

In [ ]:
# Sample size and quick preview
print('sample_df shape:', sample_df.shape)
display(sample_df.head(3))

In [ ]:
# 1) Surface distribution
if 'surface' in sample_df.columns:
    plt.figure(figsize=(6,3))
    sns.countplot(data=sample_df, x='surface', order=sample_df['surface'].value_counts().index, color='#1f77b4')
    plt.title('Surface Distribution (sample)')
    plt.xlabel('surface')
    plt.ylabel('count')
    plt.xticks(rotation=0)
    plt.show()
else:
    print('surface column not found')

In [ ]:
# 2) Surface x Tourney Level (count)
if {'surface','tourney_level'}.issubset(sample_df.columns):
    plt.figure(figsize=(8,3.5))
    order_s = sample_df['surface'].value_counts().index
    order_l = sample_df['tourney_level'].value_counts().index
    sns.countplot(data=sample_df, x='surface', hue='tourney_level', order=order_s, hue_order=order_l)
    plt.title('Surface x Tourney Level (sample)')
    plt.xlabel('surface')
    plt.ylabel('count')
    plt.legend(title='tourney_level', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
else:
    print('Required columns not found: surface, tourney_level')

In [ ]:
# 3) Match duration (minutes) - boxplot by surface
if {'minutes','surface'}.issubset(sample_df.columns):
    dfm = sample_df[['minutes','surface']].dropna().copy()
    if not dfm.empty:
        # Winsorize-like clipping to reduce outlier effect
        q1, q99 = dfm['minutes'].quantile([0.01, 0.99])
        dfm['minutes_clip'] = dfm['minutes'].clip(q1, q99)
        plt.figure(figsize=(8,3.5))
        sns.boxplot(data=dfm, x='surface', y='minutes_clip', order=dfm['surface'].value_counts().index)
        plt.title('Minutes by Surface (sample, 1-99% clipped)')
        plt.xlabel('surface')
        plt.ylabel('minutes (clip)')
        plt.show()
    else:
        print('Not enough data for minutes or surface')
else:
    print('Required columns not found: minutes, surface')

In [ ]:
# 4) Yearly trend (tourney_date -> year) within sample
if 'tourney_date' in sample_df.columns:
    years_from_date = None
    s = sample_df['tourney_date'].dropna()
    try:
        # tip: in some files YYYYMMDD may be int/str
        s = s.astype(str).str.slice(0,4).astype(int)
        years_from_date = s
    except Exception as e:
        print('Could not extract year:', e)
    if years_from_date is not None and not years_from_date.empty:
        cnt = years_from_date.value_counts().sort_index()
        plt.figure(figsize=(12,3.5))
        cnt.plot(kind='bar', color='#1f77b4')
        plt.title('Match Count by Year in Sample (tourney_date)')
        plt.xlabel('Year')
        plt.ylabel('count')
        plt.tight_layout()
        plt.show()
    else:
        print('Not enough data for year distribution')
else:
    print('tourney_date column not found')

In [ ]:
# 5) Players with most wins (sample)
if 'winner_name' in sample_df.columns:
    top_winners = sample_df['winner_name'].value_counts().head(20)
    plt.figure(figsize=(8,5))
    top_winners.sort_values().plot(kind='barh', color='#2ca02c')
    plt.title('Players with Most Wins (Top 20, sample)')
    plt.xlabel('count')
    plt.ylabel('winner_name')
    plt.tight_layout()
    plt.show()
    display(top_winners)
else:
    print('winner_name column not found')

In [ ]:
# 6) Rank difference analysis (upset rate)
rank_cols = {'winner_rank','loser_rank'}
if rank_cols.issubset(sample_df.columns):
    r = sample_df[list(rank_cols)].dropna().copy()
    if not r.empty:
        r['rank_diff'] = r['loser_rank'] - r['winner_rank']  # + if favorite won, - if upset
        plt.figure(figsize=(7,3.5))
        sns.histplot(r['rank_diff'], bins=50, color='#9467bd')
        plt.axvline(0, color='red', linestyle='--', linewidth=1)
        plt.title('Rank Difference (loser_rank - winner_rank)')
        plt.xlabel('rank_diff')
        plt.ylabel('count')
        plt.tight_layout()
        plt.show()
        upset_rate = (r['rank_diff'] < 0).mean() * 100
        print(f'Upset rate (winner rank numerically worse -> rank_diff < 0): {upset_rate:.1f}%')
    else:
        print('Not enough data for rank difference')
else:
    print('Required columns not found: winner_rank, loser_rank')

In [ ]:
# 7) Simple correlation heatmap (selected numerical columns)
num_candidates = [
    'minutes','best_of','draw_size',
    'winner_age','loser_age','winner_ht','loser_ht',
    'winner_rank','loser_rank','winner_rank_points','loser_rank_points'
]
cols_exist = [c for c in num_candidates if c in sample_df.columns]
if len(cols_exist) >= 2:
    dfc = sample_df[cols_exist].copy()
    if dfc.dropna(how='all').shape[0] > 0:
        corr = dfc.corr(numeric_only=True)
        plt.figure(figsize=(8,6))
        sns.heatmap(corr, annot=False, cmap='vlag', center=0, square=True)
        plt.title('Correlation Heatmap (sample)')
        plt.tight_layout()
        plt.show()
    else:
        print('Not enough numerical/complete data for correlation')
else:
    print('At least two numerical columns required for correlation')

## Notes
- Total row count is calculated with chunked counting (RAM friendly).
- Variable count is estimated by extracting column set from first few files.
- Missing value and distribution analyses are performed on sample; can be extended to all data with chunks if desired.
